[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_06/MFM_ch6_copula_inference/MFM_ch6_copula_inference.ipynb)

# MFM_ch6_copula_inference

Inference for copulas: the Hoeffding-projection standard error of Kendall tau, Genest-Ghoudi-Rivest sandwich standard errors for canonical maximum likelihood versus the inverse Hessian, a Monte Carlo illustration of Chen and Fan (2006) on GARCH residuals, a bootstrap test of tail symmetry, and a GAS (score-driven) t copula for JPMorgan Chase and Bank of America.

*Modelling Financial Markets (MFM), Chapter 6: Multivariate Volatility and Dependence. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_6'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy import stats, optimize, integrate
from scipy.signal import lfilter
from scipy.special import gammaln
try:
    from arch import arch_model
except ImportError:                      # Colab: pip install arch
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)
    from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
FAM_COL = {'gaussian': MainBlue, 't': IDAred, 'clayton': Forest, 'gumbel': Amber, 'frank': Purple}

SEED = 42
NUM = {}
TABLE_DIR = '.'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def shade_crises(ax, alpha=0.12):
    for a, b in (('2008-09-15', '2009-03-31'), ('2020-02-20', '2020-04-30'), ('2022-01-03', '2022-10-31')):
        ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=Gray, alpha=alpha, lw=0)

## Data

In [ ]:
import os
import numpy as np
import pandas as pd

REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, coloana, eticheta, data de start)
ASSETS = {
    'spy':   ('SPY.US',        'adjusted_close', 'SPY (S&P 500 ETF)',          '2002-07-30'),
    'tlt':   ('TLT.US',        'adjusted_close', 'TLT (20+y Treasury ETF)',    '2002-07-30'),
    'ief':   ('IEF.US',        'adjusted_close', 'IEF (7-10y Treasury ETF)',   '2002-07-30'),
    'sp500': ('GSPC.INDX',     'close',          'S&P 500',                    '2000-01-01'),
    'stoxx': ('STOXX50E.INDX', 'close',          'Euro Stoxx 50',              '2000-01-01'),
    'bet':   ('BET',           'close',          'BET (Romania)',              '2000-01-01'),
    'bettr': ('BETTR.INDX',    'close',          'BET-TR (Romania)',           '2014-09-23'),
    'btc':   ('BTC-USD.CC',    'close',          'Bitcoin',                    '2014-09-17'),
    'vix':   ('VIX.INDX',      'close',          'VIX',                        '2000-01-01'),
    'jpm':   ('JPM.US',        'adjusted_close', 'JPMorgan Chase',             '2010-01-01'),
    'bac':   ('BAC.US',        'adjusted_close', 'Bank of America',            '2010-01-01'),
    'dbk':   ('DBK.XETRA',     'adjusted_close', 'Deutsche Bank',              '2010-01-01'),
    'bnp':   ('BNP.PA',        'adjusted_close', 'BNP Paribas',                '2010-01-01'),
    'tlv':   ('TLV.RO',        'adjusted_close', 'Banca Transilvania',         '2010-01-01'),
    'brd':   ('BRD.RO',        'adjusted_close', 'BRD-Groupe Societe Generale', '2010-01-01'),
}
LABELS = {k: v[2] for k, v in ASSETS.items()}
SHORT = {'spy': 'SPY', 'tlt': 'TLT', 'ief': 'IEF', 'sp500': 'S&P 500', 'stoxx': 'Euro Stoxx 50', 'bet': 'BET',
         'bettr': 'BET-TR', 'btc': 'Bitcoin', 'vix': 'VIX', 'jpm': 'JPM', 'bac': 'BAC', 'dbk': 'DBK', 'bnp': 'BNP',
         'tlv': 'TLV', 'brd': 'BRD'}
BANKS = ['jpm', 'bac', 'dbk', 'bnp', 'tlv', 'brd']


def read_market(symbol):
    """Seria zilnica <SIMBOL> (data/market)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_price(name, start=None, end=END):
    """Pretul zilnic curatat dupa conventiile capitolului."""
    symbol, col, _, start0 = ASSETS[name]
    d = read_market(symbol).loc[start or start0:end]
    d = d[d[col] > 0].dropna(subset=[col])
    s = d[col]
    if name != 'btc':
        keep = s.index.dayofweek < 5                                   # fara cotatii de weekend
        vol = d['volume'].fillna(0) if 'volume' in d else pd.Series(0.0, index=d.index)
        keep &= ~((s.diff() == 0) & (vol <= 0)).values                # sarbatori: pret neschimbat SI volum zero
        s = s[keep]                                                    # (zilele cu volum si pret neschimbat raman)
    return s.rename(name)


def joint_prices(names, start=None, end=END):
    """Preturile activelor in zilele comune (join interior pe preturi)."""
    return pd.concat([load_price(n, start, end) for n in names], axis=1, join='inner').dropna()


def joint_returns(names, start=None, end=END):
    """Randamente log zilnice calculate DUPA join-ul pe preturi (zile comune)."""
    return np.log(joint_prices(names, start, end)).diff().dropna()


def weekly_returns(names, start=None, end=END):
    """Randamente log saptamanale: ultimul pret comun din fiecare saptamana (vineri), apoi diferente."""
    p = joint_prices(names, start, end)
    return np.log(p.resample('W-FRI').last()).diff().dropna()

## Dependence tools: EWMA, GARCH, DCC, Forbes-Rigobon, copulas

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats, optimize, integrate
from scipy.signal import lfilter
from scipy.special import gammaln
from arch import arch_model
try:
    from numba import njit
except ImportError:                      # without numba the same functions run as plain Python (slower)
    def njit(*args, **kwargs):
        return args[0] if (len(args) == 1 and callable(args[0])) else (lambda f: f)


# =============================================================================
# EWMA si fereastra mobila
# =============================================================================
def ewma_cov(R, lam=0.94, init=100):
    """Sigma_t = lam*Sigma_{t-1} + (1-lam) r_{t-1} r_{t-1}'  (prognoza pentru ziua t, fara r_t).
    Primele init observatii servesc doar la initializare: Sigma_init = covarianta lor; inainte de init, NaN."""
    X = np.asarray(R, dtype=float)
    T, N = X.shape
    S = np.full((T, N, N), np.nan)
    S[init] = np.cov(X[:init].T)
    for t in range(init + 1, T):
        S[t] = lam * S[t - 1] + (1 - lam) * np.outer(X[t - 1], X[t - 1])
    return S


def ewma_corr(R, lam=0.94, i=0, j=1):
    S = ewma_cov(R, lam)
    return pd.Series(S[:, i, j] / np.sqrt(S[:, i, i] * S[:, j, j]), index=R.index)


def rolling_corr(R, window, a=None, b=None):
    a, b = a or R.columns[0], b or R.columns[1]
    return R[a].rolling(window).corr(R[b])


def fisher_ci(rho, n, level=0.95):
    """Interval de incredere pentru corelatie prin transformarea Fisher z (date i.i.d.)."""
    z, se = np.arctanh(rho), 1 / np.sqrt(n - 3)
    q = stats.norm.ppf(0.5 + level / 2)
    return np.tanh(z - q * se), np.tanh(z + q * se)


# =============================================================================
# GARCH(1,1) univariat -- primul pas al DCC
# =============================================================================
def garch_filter(r, dist='normal'):
    """GARCH(1,1) cu medie constanta pe randamente in procente (QML).
    Intoarce parametrii, volatilitatea conditionala (%) si reziduurile standardizate."""
    res = arch_model(100 * r, mean='Constant', vol='GARCH', p=1, q=1, dist=dist).fit(disp='off')
    z = (res.resid / res.conditional_volatility).rename(r.name)
    return res.params, res.conditional_volatility.rename(r.name), z, res.loglikelihood


def garch_all(R, dist='normal'):
    out = {c: garch_filter(R[c], dist) for c in R.columns}
    Z = pd.concat([out[c][2] for c in R.columns], axis=1)
    V = pd.concat([out[c][1] for c in R.columns], axis=1)
    P = pd.DataFrame({c: out[c][0] for c in R.columns})
    ll = sum(out[c][3] for c in R.columns)
    return P, V, Z, ll


# =============================================================================
# DCC (Engle, 2002): Q_t = (1-a-b) Qbar + a z_{t-1} z_{t-1}' + b Q_{t-1}
# =============================================================================
def dcc_path(Z, a, b, Qbar=None):
    """Matricele de corelatie R_t (T x N x N). Fiecare element al lui Q_t urmeaza o recursie
    liniara, calculata vectorial cu lfilter."""
    X = np.asarray(Z, dtype=float)
    T, N = X.shape
    Qbar = np.cov(X.T, bias=True) if Qbar is None else Qbar
    P = np.einsum('ti,tj->tij', X, X)                  # z_t z_t'
    drive = np.empty_like(P)
    drive[0] = Qbar
    drive[1:] = (1 - a - b) * Qbar + a * P[:-1]         # intrare la momentul t: foloseste z_{t-1}
    Q = lfilter([1.0], [1.0, -b], drive.reshape(T, -1), axis=0).reshape(T, N, N)
    Q[0] = Qbar
    d = np.sqrt(np.einsum('tii->ti', Q))
    return Q / (d[:, :, None] * d[:, None, :])


@njit(cache=True)
def dcc2_corr_loglik(x, y, a, b, q11b, q22b, q12b):
    """Cazul bivariat al lui dcc_corr_loglik, scris ca o singura bucla (aceeasi recursie ca dcc_path)."""
    q11, q22, q12 = q11b, q22b, q12b
    c = 1.0 - a - b
    ll = 0.0
    for t in range(len(x)):
        r = q12 / np.sqrt(q11 * q22)
        d = 1.0 - r * r
        ll += -0.5 * (np.log(d) + (x[t] * x[t] + y[t] * y[t] - 2.0 * r * x[t] * y[t]) / d - x[t] * x[t] - y[t] * y[t])
        q11 = c * q11b + a * x[t] * x[t] + b * q11
        q22 = c * q22b + a * y[t] * y[t] + b * q22
        q12 = c * q12b + a * x[t] * y[t] + b * q12
    return ll


def dcc_corr_loglik(Z, a, b, Qbar=None):
    """Partea de corelatie a log-verosimilitatii: -1/2 sum(log|R_t| + z'R_t^{-1}z - z'z)."""
    X = np.asarray(Z, dtype=float)
    if X.shape[1] == 2:
        Q = np.cov(X.T, bias=True) if Qbar is None else Qbar
        return dcc2_corr_loglik(np.ascontiguousarray(X[:, 0]), np.ascontiguousarray(X[:, 1]), float(a), float(b),
                                Q[0, 0], Q[1, 1], Q[0, 1])
    R = dcc_path(X, a, b, Qbar)
    sign, logdet = np.linalg.slogdet(R)
    quad = np.einsum('ti,tij,tj->t', X, np.linalg.inv(R), X)
    return -0.5 * np.sum(logdet + quad - np.sum(X ** 2, axis=1))


def ccc_loglik(Z):
    """CCC (Bollerslev, 1990): corelatie constanta = DCC cu a = b = 0."""
    return dcc_corr_loglik(Z, 0.0, 0.0)


def _num_hessian(f, x, h=1e-4):
    k = len(x)
    H = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            e_i, e_j = np.eye(k)[i] * h, np.eye(k)[j] * h
            H[i, j] = (f(x + e_i + e_j) - f(x + e_i - e_j) - f(x - e_i + e_j) + f(x - e_i - e_j)) / (4 * h * h)
    return H


def dcc_fit(Z, start=(0.03, 0.95)):
    """Pasul 2 al DCC: (a, b) prin verosimilitate maxima, cu Qbar fixat la corelatia de selectie
    (correlation targeting). Erorile standard provin din hessiana pasului 2 (ignora eroarea pasului 1)."""
    X = np.asarray(Z, dtype=float)
    Qbar = np.cov(X.T, bias=True)

    def nll(p):
        a, b = p
        if a < 0 or b < 0 or a + b >= 0.9999:
            return 1e10
        return -dcc_corr_loglik(X, a, b, Qbar)

    best = None
    for s in (start, (0.01, 0.98), (0.05, 0.90), (0.10, 0.80)):
        r = optimize.minimize(nll, s, method='Nelder-Mead', options=dict(xatol=1e-7, fatol=1e-7, maxiter=4000))
        if best is None or r.fun < best.fun:
            best = r
    a, b = best.x
    try:
        H = _num_hessian(lambda p: -nll(p), best.x)
        se = np.sqrt(np.diag(np.linalg.inv(-H)))
    except Exception:
        se = np.array([np.nan, np.nan])
    return dict(a=a, b=b, se_a=se[0], se_b=se[1], loglik=-best.fun, Qbar=Qbar,
                R=dcc_path(X, a, b, Qbar), ccc_loglik=ccc_loglik(X))


# =============================================================================
# Forbes-Rigobon (2002)
# =============================================================================
def fr_adjust(rho_c, delta):
    """Corelatia din criza corectata pentru cresterea varianţei pietei-sursa:
    rho* = rho_c / sqrt(1 + delta (1 - rho_c^2)), delta = Var_criza/Var_calm - 1."""
    return rho_c / np.sqrt(1 + delta * (1 - rho_c ** 2))


def fr_inflation(rho, delta):
    """Corelatia masurata in criza cand corelatia adevarata nu se schimba: rho sqrt((1+delta)/(1+delta rho^2))."""
    return rho * np.sqrt((1 + delta) / (1 + delta * rho ** 2))


def fisher_z_test(rho1, n1, rho0, n0):
    """H0: rho1 <= rho0 (test unilateral, esantioane independente)."""
    z = (np.arctanh(rho1) - np.arctanh(rho0)) / np.sqrt(1 / (n1 - 3) + 1 / (n0 - 3))
    return z, 1 - stats.norm.cdf(z)


def crisis_table(ret2, source, targets, calm, crisis):
    """Corelatia calm vs criza, corectia Forbes-Rigobon si testul Fisher z pentru mai multe piete."""
    rows = []
    c0, c1 = ret2.loc[calm[0]:calm[1]], ret2.loc[crisis[0]:crisis[1]]
    delta = c1[source].var() / c0[source].var() - 1
    for tg in targets:
        r0, r1 = c0[source].corr(c0[tg]), c1[source].corr(c1[tg])
        adj = fr_adjust(r1, delta)
        n0, n1 = len(c0) // 2, len(c1) // 2          # randamente pe 2 zile suprapuse: n efectiv ~ n/2
        rows.append(dict(target=tg, rho_calm=r0, rho_crisis=r1, delta=delta, rho_adj=adj,
                         z_raw=fisher_z_test(r1, n1, r0, n0)[0], p_raw=fisher_z_test(r1, n1, r0, n0)[1],
                         z_adj=fisher_z_test(adj, n1, r0, n0)[0], p_adj=fisher_z_test(adj, n1, r0, n0)[1],
                         n_calm=len(c0), n_crisis=len(c1)))
    return pd.DataFrame(rows).set_index('target')


def exceedance_corr(x, y, qs):
    """Corelatia de depasire (Longin-Solnik, 2001): corr(x, y | x<q_x, y<q_y) pentru cozi joase si
    corr(x, y | x>q_x, y>q_y) pentru cozi inalte, la aceleasi cuantile marginale."""
    out = []
    for q in qs:
        lo = (x <= np.quantile(x, q)) & (y <= np.quantile(y, q))
        hi = (x >= np.quantile(x, 1 - q)) & (y >= np.quantile(y, 1 - q))
        out.append((q, np.corrcoef(x[lo], y[lo])[0, 1] if lo.sum() > 5 else np.nan, lo.sum(),
                    np.corrcoef(x[hi], y[hi])[0, 1] if hi.sum() > 5 else np.nan, hi.sum()))
    return pd.DataFrame(out, columns=['q', 'rho_low', 'n_low', 'rho_high', 'n_high']).set_index('q')


# =============================================================================
# Copule bivariate
# =============================================================================
FAMILIES = ['gaussian', 't', 'clayton', 'gumbel', 'frank']
FAM_LABEL = {'gaussian': 'Gaussian', 't': 'Student t', 'clayton': 'Clayton', 'gumbel': 'Gumbel', 'frank': 'Frank'}
EPS = 1e-10


def pseudo_obs(X):
    """Pseudo-observatii: rang / (n + 1), coloana cu coloana."""
    X = np.asarray(X, dtype=float)
    return np.column_stack([stats.rankdata(X[:, k]) / (len(X) + 1) for k in range(X.shape[1])])


def kendall_tau(u, v):
    return stats.kendalltau(u, v)[0]


def spearman_rho(u, v):
    return stats.spearmanr(u, v)[0]


def _debye1(x):
    if abs(x) < 1e-8:
        return 1.0
    val = integrate.quad(lambda t: t / np.expm1(t), 0, abs(x))[0] / abs(x)
    return val if x > 0 else val + abs(x) / 2


def tau_of(fam, theta):
    """Tau Kendall implicat de parametrul copulei."""
    if fam in ('gaussian', 't'):
        return 2 / np.pi * np.arcsin(theta[0] if np.ndim(theta) else theta)
    th = float(np.atleast_1d(theta)[0])
    if fam == 'clayton':
        return th / (th + 2)
    if fam == 'gumbel':
        return 1 - 1 / th
    if fam == 'frank':
        return 1 - 4 / th * (1 - _debye1(th))


def theta_from_tau(fam, tau):
    """Inversarea relatiei tau <-> parametru (metoda momentelor pe tau Kendall)."""
    if fam in ('gaussian', 't'):
        return np.sin(np.pi * tau / 2)
    if fam == 'clayton':
        return 2 * tau / (1 - tau)
    if fam == 'gumbel':
        return 1 / (1 - tau)
    if fam == 'frank':
        return optimize.brentq(lambda th: tau_of('frank', th) - tau, 1e-6 if tau > 0 else -60, 60 if tau > 0 else -1e-6)


def tail_dep(fam, par):
    """(lambda_L, lambda_U) pentru parametrii dati."""
    if fam == 'gaussian':
        return 0.0, 0.0
    if fam == 't':
        rho, nu = par
        lam = 2 * stats.t.cdf(-np.sqrt((nu + 1) * (1 - rho) / (1 + rho)), nu + 1)
        return lam, lam
    th = par[0]
    if fam == 'clayton':
        return 2 ** (-1 / th), 0.0
    if fam == 'gumbel':
        return 0.0, 2 - 2 ** (1 / th)
    return 0.0, 0.0


def log_density(fam, par, u, v):
    u = np.clip(u, EPS, 1 - EPS)
    v = np.clip(v, EPS, 1 - EPS)
    if fam == 'gaussian':
        rho = par[0]
        x, y = stats.norm.ppf(u), stats.norm.ppf(v)
        return -0.5 * np.log(1 - rho ** 2) - (rho ** 2 * (x ** 2 + y ** 2) - 2 * rho * x * y) / (2 * (1 - rho ** 2))
    if fam == 't':
        rho, nu = par
        x, y = stats.t.ppf(u, nu), stats.t.ppf(v, nu)
        return (gammaln((nu + 2) / 2) + gammaln(nu / 2) - 2 * gammaln((nu + 1) / 2) - 0.5 * np.log(1 - rho ** 2)
                - (nu + 2) / 2 * np.log1p((x ** 2 + y ** 2 - 2 * rho * x * y) / (nu * (1 - rho ** 2)))
                + (nu + 1) / 2 * (np.log1p(x ** 2 / nu) + np.log1p(y ** 2 / nu)))
    th = par[0]
    if fam == 'clayton':
        return (np.log1p(th) - (1 + th) * (np.log(u) + np.log(v))
                - (2 + 1 / th) * np.log(u ** -th + v ** -th - 1))
    if fam == 'gumbel':
        lu, lv = -np.log(u), -np.log(v)
        A = lu ** th + lv ** th
        Ath = A ** (1 / th)
        return (-Ath - np.log(u) - np.log(v) + (th - 1) * (np.log(lu) + np.log(lv))
                + (2 / th - 2) * np.log(A) + np.log1p((th - 1) / Ath))
    if fam == 'frank':
        e = -np.expm1(-th)
        den = e - (-np.expm1(-th * u)) * (-np.expm1(-th * v))
        return np.log(th * e) - th * (u + v) - 2 * np.log(np.abs(den))


def h_func(fam, par, u, v):
    """h(v|u) = dC(u,v)/du: functia de repartitie conditionata (transformata Rosenblatt)."""
    u = np.clip(u, EPS, 1 - EPS)
    v = np.clip(v, EPS, 1 - EPS)
    if fam == 'gaussian':
        rho = par[0]
        x, y = stats.norm.ppf(u), stats.norm.ppf(v)
        return stats.norm.cdf((y - rho * x) / np.sqrt(1 - rho ** 2))
    if fam == 't':
        rho, nu = par
        x, y = stats.t.ppf(u, nu), stats.t.ppf(v, nu)
        return stats.t.cdf((y - rho * x) / np.sqrt((nu + x ** 2) * (1 - rho ** 2) / (nu + 1)), nu + 1)
    th = par[0]
    if fam == 'clayton':
        return u ** (-th - 1) * (u ** -th + v ** -th - 1) ** (-1 / th - 1)
    if fam == 'gumbel':
        lu, lv = -np.log(u), -np.log(v)
        A = lu ** th + lv ** th
        return np.exp(-A ** (1 / th)) / u * lu ** (th - 1) * A ** (1 / th - 1)
    if fam == 'frank':
        a, b = np.expm1(-th * u), np.expm1(-th * v)
        return (a + 1) * b / (np.expm1(-th) + a * b)


def simulate(fam, par, n, rng):
    """Simulare de n perechi (u, v) din copula."""
    if fam == 'gaussian':
        rho = par[0]
        z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
        return stats.norm.cdf(z)
    if fam == 't':
        rho, nu = par
        z = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], n)
        w = np.sqrt(nu / rng.chisquare(nu, n))
        return stats.t.cdf(z * w[:, None], nu)
    th = par[0]
    if fam == 'clayton':          # Marshall-Olkin cu factor Gamma(1/theta)
        V = rng.gamma(1 / th, 1, n)
        E = rng.exponential(1, (n, 2))
        return (1 + E / V[:, None]) ** (-1 / th)
    if fam == 'gumbel':           # Marshall-Olkin cu factor stabil pozitiv, alpha = 1/theta (Kanter)
        al = 1 / th
        U = rng.uniform(0, np.pi, n)
        W = rng.exponential(1, n)
        S = np.sin(al * U) / np.sin(U) ** (1 / al) * (np.sin((1 - al) * U) / W) ** ((1 - al) / al)
        E = rng.exponential(1, (n, 2))
        return np.exp(-(E / S[:, None]) ** al)
    if fam == 'frank':            # inversarea functiei h
        u, w = rng.uniform(size=n), rng.uniform(size=n)
        v = -np.log1p(w * np.expm1(-th) / (w + (1 - w) * np.exp(-th * u))) / th
        return np.column_stack([u, v])


def fit_copula(fam, u, v):
    """Estimare ML pe pseudo-observatii (pseudo-verosimilitate canonica, CML)."""
    tau = kendall_tau(u, v)
    if fam == 'gaussian':
        f = lambda p: -np.sum(log_density('gaussian', [np.tanh(p[0])], u, v))
        r = optimize.minimize(f, [np.arctanh(np.sin(np.pi * tau / 2))], method='BFGS')
        par = [np.tanh(r.x[0])]
    elif fam == 't':
        f = lambda p: -np.sum(log_density('t', [np.tanh(p[0]), 2.01 + np.exp(p[1])], u, v))
        best = None
        for nu0 in (3, 6, 15):
            r = optimize.minimize(f, [np.arctanh(np.sin(np.pi * tau / 2)), np.log(nu0 - 2.01)], method='Nelder-Mead',
                                  options=dict(xatol=1e-6, fatol=1e-6))
            if best is None or r.fun < best.fun:
                best = r
        r = best
        par = [np.tanh(r.x[0]), 2.01 + np.exp(r.x[1])]
    else:
        lo, hi = {'clayton': (1e-4, 30), 'gumbel': (1.0001, 30), 'frank': (1e-4, 60)}[fam]
        f = lambda p: -np.sum(log_density(fam, [p], u, v))
        th0 = min(max(theta_from_tau(fam, max(tau, 0.02)), lo * 1.01), hi * 0.99)
        r = optimize.minimize_scalar(f, bounds=(lo, hi), method='bounded', options=dict(xatol=1e-7))
        if f(th0) < r.fun:
            r = optimize.minimize_scalar(f, bracket=(lo * 1.01, th0, hi * 0.99), bounds=(lo, hi), method='bounded')
        par = [r.x]
    ll = np.sum(log_density(fam, par, u, v))
    k = 2 if fam == 't' else 1
    lamL, lamU = tail_dep(fam, par)
    return dict(family=fam, par=list(map(float, par)), loglik=float(ll), aic=float(2 * k - 2 * ll),
                bic=float(k * np.log(len(u)) - 2 * ll), tau_model=float(tau_of(fam, par if fam in ('gaussian', 't') else par[0])),
                lamL=float(lamL), lamU=float(lamU))


def cvm_rosenblatt(fam, par, u, v, chunk=1500):
    """S_n^(B) (Genest, Remillard & Beaudoin, 2009): distanta Cramer-von Mises dintre repartitia
    empirica a perechilor Rosenblatt (u, h(v|u)) si copula de independenta."""
    E = np.column_stack([u, h_func(fam, par, u, v)])
    n = len(E)
    t1 = n / 9
    t2 = np.sum((1 - E[:, 0] ** 2) * (1 - E[:, 1] ** 2)) / 2
    t3 = 0.0
    for i in range(0, n, chunk):
        Ei = E[i:i + chunk]
        t3 += np.sum((1 - np.maximum(Ei[:, None, 0], E[None, :, 0])) * (1 - np.maximum(Ei[:, None, 1], E[None, :, 1])))
    return t1 - t2 + t3 / n


def gof_test(fam, u, v, B=200, seed=42, fit=None):
    """Test de adecvare cu bootstrap parametric: simulare din copula estimata, pseudo-observatii,
    re-estimare, statistica S_n^(B); p-valoarea = proportia statisticilor bootstrap >= cea observata."""
    fit = fit or fit_copula(fam, u, v)
    s0 = cvm_rosenblatt(fam, fit['par'], u, v)
    rng = np.random.default_rng(seed)
    sb = []
    for _ in range(B):
        X = simulate(fam, fit['par'], len(u), rng)
        U = pseudo_obs(X)
        fb = fit_copula(fam, U[:, 0], U[:, 1])
        sb.append(cvm_rosenblatt(fam, fb['par'], U[:, 0], U[:, 1]))
    sb = np.array(sb)
    return dict(stat=float(s0), p=float((1 + np.sum(sb >= s0)) / (B + 1)), B=B)


def empirical_tail_dep(u, v, q):
    """lambda_L(q) = P(U<=q, V<=q)/q si lambda_U(q) = P(U>1-q, V>1-q)/q."""
    return np.mean((u <= q) & (v <= q)) / q, np.mean((u > 1 - q) & (v > 1 - q)) / q

## Helper functions

In [ ]:
N = {}

def bank_copula_data():
    R = joint_returns(['jpm', 'bac'])
    P, V, Z, ll = garch_all(R)
    U = pseudo_obs(Z.values)
    return R, Z, U


def stationary_bootstrap_idx(n, mean_block, rng):
    """Indicii unui bootstrap stationar (Politis-Romano): blocuri de lungime geometrica."""
    idx = np.empty(n, dtype=int)
    idx[0] = rng.integers(n)
    for t in range(1, n):
        idx[t] = rng.integers(n) if rng.uniform() < 1 / mean_block else (idx[t - 1] + 1) % n
    return idx

## Analysis

In [ ]:
def emp_copula_at_points(u, v, chunk=2000):
    n = len(u)
    out = np.empty(n)
    for i in range(0, n, chunk):
        out[i:i + chunk] = np.mean((u[None, :] <= u[i:i + chunk, None]) & (v[None, :] <= v[i:i + chunk, None]), axis=1)
    return out


def tau_se(u, v):
    """Eroarea standard a lui tau Kendall din proiectia Hoeffding: 4 sd(2 C_n(U, V) - U - V) / sqrt(n)."""
    W = 2 * emp_copula_at_points(u, v) - u - v
    return kendall_tau(u, v), 4 * W.std() / np.sqrt(len(u))


def cml_se(fam, par, u, v, hp=1e-5, hu=1e-6):
    """Erori standard CML: inversa hessienei (margini cunoscute) si sandwich-ul Genest-Ghoudi-Rivest (1995),
    in care scorul primeste termenii W1(U_i) + W2(V_i) datorati rangurilor."""
    par = np.asarray(par, float)
    k, n = len(par), len(u)

    def score(p, uu, vv):
        g = np.empty((len(uu), k))
        for j in range(k):
            e = np.zeros(k)
            e[j] = hp * max(1, abs(p[j]))
            g[:, j] = (log_density(fam, p + e, uu, vv) - log_density(fam, p - e, uu, vv)) / (2 * e[j])
        return g
    S = score(par, u, v)
    A = np.zeros((k, k))
    for j in range(k):
        e = np.zeros(k)
        e[j] = 1e-4 * max(1, abs(par[j]))
        A[:, j] = -(score(par + e, u, v) - score(par - e, u, v)).mean(0) / (2 * e[j])
    A = (A + A.T) / 2
    du = (score(par, u + hu, v) - score(par, u - hu, v)) / (2 * hu)
    dv = (score(par, u, v + hu) - score(par, u, v - hu)) / (2 * hu)

    def suffix(x, D):
        o = np.argsort(x)
        cs = np.cumsum(D[o][::-1], axis=0)[::-1] / n
        out = np.empty_like(D)
        out[o] = cs
        return out
    Ai = np.linalg.inv(A)
    Bg = np.atleast_2d(np.cov((S + suffix(u, du) + suffix(v, dv)).T))
    return np.sqrt(np.diag(Ai) / n), np.sqrt(np.diag(Ai @ Bg @ Ai) / n)


def simulate_garch(mu, omega, alpha, beta, eps):
    """Randamente (in %) dintr-un GARCH(1,1) cu medie constanta si inovatiile standardizate eps."""
    T = len(eps)
    r = np.empty(T)
    s2 = omega / (1 - alpha - beta)
    for t in range(T):
        r[t] = mu + np.sqrt(s2) * eps[t]
        s2 = omega + alpha * (r[t] - mu) ** 2 + beta * s2
    return r


def rank_inference():
    Rb, Zb, U = bank_copula_data()
    u, v = U[:, 0], U[:, 1]
    tau, se_tau = tau_se(u, v)
    rho_tau = np.sin(np.pi * tau / 2)
    se_rho_tau = np.pi / 2 * np.cos(np.pi * tau / 2) * se_tau
    ft = fit_copula('t', u, v)
    naive, ggr = cml_se('t', ft['par'], u, v)
    N.update(ri_tau=tau, ri_se_tau=se_tau, ri_rho_tau=rho_tau, ri_se_rho_tau=se_rho_tau, ri_rho_cml=ft['par'][0],
             ri_nu_cml=ft['par'][1], ri_se_rho_naive=naive[0], ri_se_rho_ggr=ggr[0], ri_se_nu_naive=naive[1],
             ri_se_nu_ggr=ggr[1], ri_n=len(u))
    # asimetria in cozi: lambda_L(q) - lambda_U(q), bootstrap i.i.d. pe perechi de reziduuri (reranguite)
    rng = np.random.default_rng(SEED)
    X = Zb.values
    for q, tag in ((0.05, '05'), (0.01, '01')):
        L0, U0 = empirical_tail_dep(u, v, q)
        db = []
        for _ in range(999):
            Ub = pseudo_obs(X[rng.integers(0, len(X), len(X))])
            lb, ub = empirical_tail_dep(Ub[:, 0], Ub[:, 1], q)
            db.append(lb - ub)
        db = np.array(db)
        N.update({f'ts_diff{tag}': L0 - U0, f'ts_se{tag}': db.std(ddof=1), f'ts_lo{tag}': np.percentile(db, 2.5),
                  f'ts_hi{tag}': np.percentile(db, 97.5), f'ts_k{tag}': int(round(q * len(u)))})
    # S&P 500 / Euro Stoxx 50 saptamanal (seminarul, A2 si B2)
    W = weekly_returns(['sp500', 'stoxx'], start='2000-01-01')
    U2 = pseudo_obs(W.values)
    u2, v2 = U2[:, 0], U2[:, 1]
    t2, se2 = tau_se(u2, v2)
    rs_impl = lambda t: 6 / np.pi * np.arcsin(np.sin(np.pi * t / 2) / 2)
    diff = rs_impl(t2) - spearman_rho(u2, v2)
    bs = []
    for _ in range(1999):
        i = rng.integers(0, len(W), len(W))
        Ub = pseudo_obs(W.values[i])
        tb = kendall_tau(Ub[:, 0], Ub[:, 1])
        bs.append(rs_impl(tb) - spearman_rho(Ub[:, 0], Ub[:, 1]))
    bs = np.array(bs)
    fg, ft2 = fit_copula('gaussian', u2, v2), fit_copula('t', u2, v2)
    ng, gg = cml_se('gaussian', fg['par'], u2, v2)
    nt, gt = cml_se('t', ft2['par'], u2, v2)
    N.update(a2_se_tau=se2, a2_se_clayton=2 / (1 - t2) ** 2 * se2, a2_se_gauss=np.pi / 2 * np.cos(np.pi * t2 / 2) * se2,
             a2_W_sd=se2 * np.sqrt(len(W)) / 4, a2_diff=diff, a2_diff_se=bs.std(ddof=1), a2_diff_lo=np.percentile(bs, 2.5),
             a2_diff_hi=np.percentile(bs, 97.5), b2_se_rho_g_naive=ng[0], b2_se_rho_g_ggr=gg[0], b2_se_rho_t_naive=nt[0],
             b2_se_rho_t_ggr=gt[0], b2_se_nu_naive=nt[1], b2_se_nu_ggr=gt[1])


def chen_fan_mc(R_=500, T=2000, rho=0.8, df=5):
    Rb, Zb, U = bank_copula_data()
    P, _, _, _ = garch_all(Rb)
    rng = np.random.default_rng(SEED)
    est_true, est_res, se_naive, se_ggr = [], [], [], []
    for _ in range(R_):
        Uc = simulate('gaussian', [rho], T, rng)
        eps = stats.t.ppf(Uc, df) / np.sqrt(df / (df - 2))
        Ut = pseudo_obs(eps)
        est_true.append(fit_copula('gaussian', Ut[:, 0], Ut[:, 1])['par'][0])
        sim = {c: simulate_garch(P.loc['mu', c], P.loc['omega', c], P.loc['alpha[1]', c], P.loc['beta[1]', c], eps[:, k])
               for k, c in enumerate(Rb.columns)}
        _, _, Zs, _ = garch_all(pd.DataFrame(sim) / 100)
        Ur = pseudo_obs(Zs.values)
        f = fit_copula('gaussian', Ur[:, 0], Ur[:, 1])
        est_res.append(f['par'][0])
        a, b = cml_se('gaussian', f['par'], Ur[:, 0], Ur[:, 1])
        se_naive.append(a[0])
        se_ggr.append(b[0])
    N.update(cf_sd_true=np.std(est_true, ddof=1), cf_sd_res=np.std(est_res, ddof=1), cf_se_naive=np.mean(se_naive),
             cf_se_ggr=np.mean(se_ggr), cf_theory_ggr=(1 - rho ** 2) / np.sqrt(T),
             cf_theory_naive=(1 - rho ** 2) / np.sqrt((1 + rho ** 2) * T), cf_R=R_, cf_T=T, cf_rho=rho, cf_df=df,
             cf_bias_res=np.mean(est_res) - rho)


def t_score_rho(rho, nu, x, y):
    """d log c_t / d rho pentru copula t (x, y = cuantilele t_nu ale pseudo-observatiilor)."""
    Q = x * x + y * y - 2 * rho * x * y
    D = nu * (1 - rho ** 2)
    dQD = (-2 * x * y * (1 - rho ** 2) + 2 * rho * Q) / (nu * (1 - rho ** 2) ** 2)
    return rho / (1 - rho ** 2) - (nu + 2) / 2 * dQD / (1 + Q / D)


@njit(cache=True)
def gas_t_nll(om, A, Bp, nu, x, y, cst):
    """Minus log-verosimilitatea copulei t GAS (aceeasi recursie ca gas_filter); x, y = cuantilele t_nu."""
    f = om / (1.0 - Bp)
    ll = 0.0
    for t in range(len(x)):
        r = np.tanh(f)
        if abs(r) >= 0.9999:
            return 1e10
        Q = x[t] * x[t] + y[t] * y[t] - 2.0 * r * x[t] * y[t]
        D = nu * (1.0 - r * r)
        ll += (cst - 0.5 * np.log(1.0 - r * r) - (nu + 2.0) / 2.0 * np.log1p(Q / D)
               + (nu + 1.0) / 2.0 * (np.log1p(x[t] * x[t] / nu) + np.log1p(y[t] * y[t] / nu)))
        dQD = (-2.0 * x[t] * y[t] * (1.0 - r * r) + 2.0 * r * Q) / (nu * (1.0 - r * r) ** 2)
        s = (r / (1.0 - r * r) - (nu + 2.0) / 2.0 * dQD / (1.0 + Q / D)) * (1.0 - r * r)
        f = om + A * s + Bp * f
    return -ll


def gas_filter(theta, u, v):
    om, A, Bp, nu = theta
    x, y = stats.t.ppf(u, nu), stats.t.ppf(v, nu)
    T = len(u)
    f = np.empty(T)
    f[0] = om / (1 - Bp)
    for t in range(T - 1):
        r = np.tanh(f[t])
        s = t_score_rho(r, nu, x[t], y[t]) * (1 - r * r)
        f[t + 1] = om + A * s + Bp * f[t]
    return np.tanh(f)


def gas_t_copula(u, v):
    fs = fit_copula('t', u, v)
    r0, nu0 = fs['par']

    uc, vc = np.clip(u, 1e-10, 1 - 1e-10), np.clip(v, 1e-10, 1 - 1e-10)

    def nll(p):
        om, A, Bp, lnu = p
        nu = 2.01 + np.exp(lnu)
        if not (0 <= Bp < 0.9995) or A < 0 or A > 1:
            return 1e10
        cst = gammaln((nu + 2) / 2) + gammaln(nu / 2) - 2 * gammaln((nu + 1) / 2)
        return gas_t_nll(om, A, Bp, nu, stats.t.ppf(uc, nu), stats.t.ppf(vc, nu), cst)
    best = None
    for Bp, A in ((0.98, 0.03), (0.95, 0.05), (0.99, 0.01)):
        p0 = [(1 - Bp) * np.arctanh(r0), A, Bp, np.log(nu0 - 2.01)]
        r = optimize.minimize(nll, p0, method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-6, maxiter=4000))
        if best is None or r.fun < best.fun:
            best = r
    om, A, Bp, lnu = best.x
    nu = 2.01 + np.exp(lnu)
    rho = gas_filter((om, A, Bp, nu), u, v)
    return dict(om=om, A=A, B=Bp, nu=nu, rho=rho, ll=-best.fun, ll_static=fs['loglik'], rho_static=r0, nu_static=nu0,
                lr=2 * (-best.fun - fs['loglik']))


def fig_gas_copula():
    Rb, Zb, U = bank_copula_data()
    g = gas_t_copula(U[:, 0], U[:, 1])
    rho = pd.Series(g['rho'], index=Zb.index)
    lam = pd.Series(2 * stats.t.cdf(-np.sqrt((g['nu'] + 1) * (1 - rho) / (1 + rho)), g['nu'] + 1), index=rho.index)
    lam_s = 2 * stats.t.cdf(-np.sqrt((g['nu_static'] + 1) * (1 - g['rho_static']) / (1 + g['rho_static'])),
                            g['nu_static'] + 1)
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    shade_crises(ax)
    ax.plot(rho.index, rho, color=IDAred, lw=0.9, label='GAS t copula: correlation rho_t')
    ax.plot(lam.index, lam, color=MainBlue, lw=0.9, label='Implied tail dependence lambda_t')
    ax.axhline(g['rho_static'], color=IDAred, ls='--', lw=0.8, label='Static t copula rho')
    ax.axhline(lam_s, color=MainBlue, ls='--', lw=0.8, label='Static t copula lambda')
    ax.set_ylabel('JPM-BAC dependence')
    ax.set_ylim(0, 1)
    legend_outside_bottom(ax, ncol=2)
    save_fig('ch6_gas_copula')
    N.update(gas_om=g['om'], gas_A=g['A'], gas_B=g['B'], gas_nu=g['nu'], gas_ll=g['ll'], gas_ll_static=g['ll_static'],
             gas_lr=g['lr'], gas_rho_min=rho.min(), gas_rho_min_date=str(rho.idxmin().date()), gas_rho_max=rho.max(),
             gas_rho_max_date=str(rho.idxmax().date()), gas_rho_last=rho.iloc[-1], gas_rho_static=g['rho_static'],
             gas_nu_static=g['nu_static'], gas_lam_min=lam.min(), gas_lam_max=lam.max(), gas_lam_static=lam_s,
             gas_rho_2020=rho.loc['2020-03-01':'2020-06-30'].mean(), gas_rho_2019=rho.loc['2019'].mean())
    # seminarul, B6: S&P 500 / Euro Stoxx 50 saptamanal
    W = weekly_returns(['sp500', 'stoxx'], start='2000-01-01')
    U2 = pseudo_obs(W.values)
    g2 = gas_t_copula(U2[:, 0], U2[:, 1])
    r2 = pd.Series(g2['rho'], index=W.index)
    N.update(b6_A=g2['A'], b6_B=g2['B'], b6_nu=g2['nu'], b6_lr=g2['lr'], b6_rho_min=r2.min(),
             b6_rho_min_date=str(r2.idxmin().date()), b6_rho_max=r2.max(), b6_rho_max_date=str(r2.idxmax().date()),
             b6_rho_static=g2['rho_static'], b6_nu_static=g2['nu_static'], b6_om=g2['om'])
    return g


def tail_symmetry_block(q_list=((0.05, '05'), (0.01, '01')), B=999):
    """lambda_L(q) - lambda_U(q) la prag fixat q, cu bootstrap stationar pe blocuri (Politis & Romano, 1994); lungimea
    medie a blocului: regula automata a lui Politis & White (2004), aplicata indicatorului 1{colt inferior} - 1{colt superior}."""
    from arch.bootstrap import optimal_block_length
    Rb, Zb, U = bank_copula_data()
    u, v = U[:, 0], U[:, 1]
    X = Zb.values
    rng = np.random.default_rng(SEED)
    for q, tag in q_list:
        ind = ((u <= q) & (v <= q)).astype(float) - ((u > 1 - q) & (v > 1 - q)).astype(float)
        blk = max(1.0, float(optimal_block_length(ind)['stationary'].iloc[0]))
        L0, U0 = empirical_tail_dep(u, v, q)
        db = []
        for _ in range(B):
            Ub = pseudo_obs(X[stationary_bootstrap_idx(len(X), blk, rng)])
            lb, ub = empirical_tail_dep(Ub[:, 0], Ub[:, 1], q)
            db.append(lb - ub)
        db = np.array(db)
        ac = [np.corrcoef(ind[:-l], ind[l:])[0, 1] for l in range(1, 11)]
        N.update({f'tsb_blk{tag}': blk, f'tsb_lo{tag}': np.percentile(db, 2.5), f'tsb_hi{tag}': np.percentile(db, 97.5),
                  f'tsb_se{tag}': db.std(ddof=1), f'tsb_acmax{tag}': float(np.max(np.abs(ac)))})
    N['tsb_B'] = B


def gas_null_bootstrap(B=199):
    """Seminarul, B6: p-valoarea raportului de verosimilitate GAS contra t static, prin bootstrap parametric sub copula t
    statica estimata (n perechi simulate, transformate in pseudo-observatii, ambele modele reestimate)."""
    W = weekly_returns(['sp500', 'stoxx'], start='2000-01-01')
    U2 = pseudo_obs(W.values)
    g = gas_t_copula(U2[:, 0], U2[:, 1])
    lr_obs = g['lr']
    rng = np.random.default_rng(SEED)
    lrs = []
    for _ in range(B):
        Us = pseudo_obs(simulate('t', [g['rho_static'], g['nu_static']], len(U2), rng))
        lrs.append(gas_t_copula(Us[:, 0], Us[:, 1])['lr'])
    lrs = np.array(lrs)
    N.update(b6g_lr_obs=lr_obs, b6g_lr_null_q95=np.percentile(lrs, 95), b6g_lr_null_max=lrs.max(),
             b6g_p_boot=(1 + np.sum(lrs >= lr_obs)) / (B + 1), b6g_B=B)

## Run

In [ ]:
rank_inference()
fig_gas_copula()
chen_fan_mc(R_=100)   # 500 in the lecture; fewer here for speed
tail_symmetry_block(B=199)   # 999 in the lecture
gas_null_bootstrap(B=49)   # 199 in the seminar
{k: v for k, v in N.items() if k.startswith(('ri_', 'ts_d', 'ts_l', 'ts_h', 'tsb', 'gas', 'cf_', 'b6g'))}

![ch6_gas_copula](./ch6_gas_copula.png)

Output: `ch6_gas_copula.pdf`